In [1]:
!pip install flask pyngrok --ignore-installed blinker

  Using cached werkzeug-3.0.4-py3-none-any.whl.metadata (3.7 kB)
  Using cached jinja2-3.1.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached click-8.1.7-py3-none-any.whl.metadata (3.0 kB)
  Using cached PyYAML-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.1 kB)
  Using cached MarkupSafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 2.3 MB/s eta 0:00:00
Using cached click-8.1.7-py3-none-any.whl (97 kB)
Using cached jinja2-3.1.4-py3-none-any.whl (133 kB)
Using cached PyYAML-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (751 kB)
Using cached werkzeug-3.0.4-py3-none-any.whl (227 kB)
Using cached MarkupSafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (25 kB)


In [2]:
!ngrok authtoken 2n7Hel5dP72VnwxIhfEixB6JgZV_464RqmQLYvhLQYgzTb1cr

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [3]:
# Required Imports
import os
import sys
import numpy as np
import tensorflow as tf
from keras.models import load_model
from keras.preprocessing.text import tokenizer_from_json
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import json
from flask import Flask, request, jsonify, render_template_string
from pyngrok import ngrok

In [4]:
# Initialize Flask App
app = Flask(__name__)

In [5]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive Mounted Successfully.")

sys.path.insert(0, '/content/drive/MyDrive/CSCK507_EMA_GroupC_Final')
import config

model_file = os.path.join(config.MAIN_DIR_PATH, config.SEQ2SEQ_NA_MODEL)
tokenizer_file = os.path.join(config.MAIN_DIR_PATH, config.TOKENIZER_FILE)
test_data_file = os.path.join(config.MAIN_DIR_PATH, config.TEST_DATA)

print(f"Model File Path: {model_file}")
print(f"Tokenizer File Path: {tokenizer_file}")
print(f"Test Data File Path: {test_data_file}")

Mounted at /content/drive
Google Drive Mounted Successfully.
Model File Path: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/seq2seq_no_attention_model.h5
Tokenizer File Path: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/tokenizer.json
Test Data File Path: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/test_data.npz


In [6]:
# Load the Pre-trained Model
seq2seq_model = load_model(model_file)
print("Model loaded successfully!")

# Print the model summary to verify its structure
seq2seq_model.summary()

Model loaded successfully!
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 Encoder-Input (InputLayer)  [(None, 30)]                 0         []                            
                                                                                                  
 Decoder-Input (InputLayer)  [(None, 30)]                 0         []                            
                                                                                                  
 Encoder-Embedding (Embeddi  (None, 30, 64)               4099366   ['Encoder-Input[0][0]']       
 ng)                                                      4                                       
                                                                                                  
 Decoder-Embedding (Embeddi  (None, 30, 64)               4099366  

In [7]:
# Load the tokenizer
with open(tokenizer_file, 'r') as f:
    tokenizer_data = json.load(f)
    tokenizer = tf.keras.preprocessing.text.tokenizer_from_json(tokenizer_data)

# Create index mappings using the correct word-level index
target_token_index = tokenizer.word_index
reverse_target_word_index = {i: word for word, i in target_token_index.items()}  # Ensure word-level mapping

print(f"Tokenizer Loaded. Number of Tokens: {len(target_token_index)}")

Tokenizer Loaded. Number of Tokens: 640525


In [9]:
# Extract Encoder and Decoder Models from the Main Model

# Extract Encoder Inputs and States
encoder_inputs = seq2seq_model.input[0]
encoder_lstm_layer = seq2seq_model.get_layer("Encoder-LSTM")
encoder_outputs, state_h_enc, state_c_enc = encoder_lstm_layer.output

# Create the Encoder Model using inputs and state outputs
encoder_model = tf.keras.Model(inputs=encoder_inputs, outputs=[state_h_enc, state_c_enc])
print("Encoder Model Created Successfully.")

# Extract Decoder Inputs and States
decoder_inputs = seq2seq_model.input[1]

# Add Decoder Embedding Layer
decoder_embedding_layer = seq2seq_model.get_layer("Decoder-Embedding")
decoder_embedding_outputs = decoder_embedding_layer(decoder_inputs)

# Prepare Initial States for the Decoder LSTM with bfloat16 precision
decoder_state_input_h = tf.keras.Input(shape=(64,), name='input_3', dtype='bfloat16')  # Hidden state input for decoder
decoder_state_input_c = tf.keras.Input(shape=(64,), name='input_4', dtype='bfloat16')  # Cell state input for decoder

# Reuse the existing Decoder LSTM and Dense layers
decoder_lstm_layer = seq2seq_model.get_layer("Decoder-LSTM")
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm_layer(
    decoder_embedding_outputs, initial_state=[decoder_state_input_h, decoder_state_input_c])

decoder_dense_layer = seq2seq_model.get_layer("Decoder-Output")
decoder_outputs = decoder_dense_layer(decoder_outputs)

# Create the Decoder Model using inputs and state outputs
decoder_model = tf.keras.Model([decoder_inputs] + [decoder_state_input_h, decoder_state_input_c], [decoder_outputs, state_h_dec, state_c_dec])
print("Decoder Model Created Successfully.")

Encoder Model Created Successfully.
Decoder Model Created Successfully.


In [10]:
def greedy_decode(input_seq, encoder_model, decoder_model, target_token_index, reverse_target_word_index, max_decoder_seq_length=30):
    states_value = encoder_model.predict(input_seq)
    start_token_index = target_token_index.get('<SOS>', 0)
    target_seq = np.array([[start_token_index]])

    decoded_sentence = []

    for _ in range(max_decoder_seq_length):
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index, '')

        if sampled_word == '<EOS>':
            break

        decoded_sentence.append(sampled_word)
        target_seq = np.array([[sampled_token_index]])
        states_value = [h, c]

    return ' '.join(decoded_sentence)

In [ ]:
# HTML Template for the Chatbot UI
html_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Chatbot UI</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            max-width: 600px;
            margin: auto;
        }
        #chat-box {
            border: 1px solid #ccc;
            height: 400px;
            overflow-y: scroll;
            padding: 10px;
            margin-bottom: 10px;
        }
        .user-message, .bot-message {
            margin: 10px 0;
            padding: 8px;
            border-radius: 8px;
        }
        .user-message {
            text-align: right;
            background-color: #e1f5fe;
        }
        .bot-message {
            text-align: left;
            background-color: #e8eaf6;
        }
    </style>
</head>
<body>
    <h2>Chat with the Bot</h2>
    <div id="chat-box"></div>
    <input type="text" id="user-input" placeholder="Type your message here..." style="width: 80%;">
    <button onclick="sendMessage()">Send</button>

    <script>
        function sendMessage() {
            const userInput = document.getElementById('user-input').value;
            if (!userInput) {
                return;
            }

            // Append the user's message to the chat-box
            const chatBox = document.getElementById('chat-box');
            const userMessageDiv = document.createElement('div');
            userMessageDiv.className = 'user-message';
            userMessageDiv.textContent = userInput;
            chatBox.appendChild(userMessageDiv);

            // Clear the input field
            document.getElementById('user-input').value = '';

            // Make the request to the backend
            fetch('/chat', {
                method: 'POST',
                headers: {
                    'Content-Type': 'application/json'
                },
                body: JSON.stringify({ input: userInput })
            })
            .then(response => response.json())
            .then(data => {
                // Append the bot's response to the chat-box
                const botMessageDiv = document.createElement('div');
                botMessageDiv.className = 'bot-message';
                botMessageDiv.textContent = data.response;
                chatBox.appendChild(botMessageDiv);

                // Scroll to the bottom of the chat-box
                chatBox.scrollTop = chatBox.scrollHeight;
            })
            .catch(error => {
                console.error('Error:', error);
            });
        }
    </script>
</body>
</html>
"""

# Flask Endpoint to Serve the Chatbot UI
@app.route('/')
def index():
    return render_template_string(html_template)

# Flask Endpoint to Handle User Input
@app.route('/chat', methods=['POST'])
def chat():
    user_input = request.json.get('input')
    if not user_input:
        return jsonify({'error': 'No input provided'}), 400

    input_tokens = tokenizer.texts_to_sequences([user_input])
    input_seq = tf.keras.preprocessing.sequence.pad_sequences(input_tokens, maxlen=30, padding='post')
    response = greedy_decode(input_seq, encoder_model, decoder_model, target_token_index, reverse_target_word_index, max_decoder_seq_length=30)

    return jsonify({'response': response})

# Create a public URL using ngrok
public_url = ngrok.connect(5000)
print(f"Public URL: {public_url}")

# Run the Flask App
if __name__ == "__main__":
    app.run(port=5000)

Public URL: NgrokTunnel: "https://f051-34-122-68-81.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 16:11:14] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 16:11:14] "GET /favicon.ico HTTP/1.1" 404 -


1/1 [==============================] - 0s 139ms/step


INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 16:11:51] "POST /chat HTTP/1.1" 200 -


1/1 [==============================] - 0s 139ms/step


...

INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 17:24:57] "POST /chat HTTP/1.1" 200 -


1/1 [==============================] - 0s 138ms/step


INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 17:26:22] "POST /chat HTTP/1.1" 200 -


1/1 [==============================] - 0s 137ms/step


INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 17:34:51] "POST /chat HTTP/1.1" 200 -
